# 📊 Keltner reversion — intraday backtest & optimize

**The idea.** High-volatility stocks overshoot: a close more than `band_mult` ATRs **below** its `band_window`-minute SMA midline (the lower **Keltner band**) while still **above** its slower `trend_window`-minute SMA tends to snap back toward the mean — the classic short-term-reversal effect, strongest in volatile names and stressed moments, interpreted as payment for providing liquidity. The position exits when the close touches the midline again, after a `horizon`-minute time stop, or at an optional `sl_mult`-ATR stop-loss; there is deliberately no trailing stop (the bet is a return *to a level*). Entries are blocked in the first `skip_open` / last `skip_close` minutes of each session; `eod_exit` chooses between strictly-intraday (`true`) and letting deep dips ride overnight (`false`, the tuned default — a large part of the reversion arrives with the next day's gap). Long-only by default — `direction: both` also fades rallies in downtrends. It is the style complement of this repo's two momentum strategies. Full thesis in [README.md](./README.md).

**The universe.** It trades `HighOctaneTop20`: the sibling repo's **HighOctane** universe (top volatility tercile within sector, imported with `tsim import-universes`), narrowed to the **20 most liquid names by trailing-90-day average dollar volume** (`tsim subset --universe HighOctane --top 20`) — volatility supplies the overshoots, liquidity makes the 1-minute fills and per-leg costs honest, and the pinned list in `config/universes.yaml` keeps runs reproducible.

**Train vs holdout.** The `data:` block sets `end` to a deliberate **train cutoff** — the sweep and the optimizer only ever see data up to it. §11 replays the picked parameters on the untouched ~2-month tail, the one number in this notebook the optimizer could not have fit. Every knob lives in [`config.yaml`](./config.yaml) (defaults, sweep grid, optimization space) — this notebook only *drives* the shared engine: **load → overview → backtest → inspect → sweep → optimize → boundary-check → verify → out-of-sample**. The same steps work for every strategy in this repo.

## 1 · Load the strategy and its data

In [ ]:
# If this import fails: select the "trading-simple" Jupyter kernel
# (see README "Quick start"), or run  uv sync --extra notebook
import warnings
warnings.filterwarnings("ignore")

from trading_simple.config import load_settings
from trading_simple.db import make_engine

settings = load_settings()
engine = make_engine(settings)
print("DB:", f"{settings.db.host}:{settings.db.port}/{settings.db.name}")

from trading_simple import backtest as bt, jobs, optimize as opt, strategies
from trading_simple.data import load_panel, symbols_with_bars
from trading_simple import plots

strat = strategies.load("keltner_reversion")
print("data :", strat.data)
print("params:", strat.params)

In [ ]:
d = strat.data
syms = symbols_with_bars(engine, settings.universe_symbols(d["universe"]),
                         d["bar"], d["start"], d.get("end"), limit=d.get("max_symbols"))
panel = load_panel(engine, syms, d["bar"], d["start"], d.get("end"),
                   session=d.get("session", "regular"), exchanges=settings.exchanges)
print(f"panel: {panel.close.shape[0]:,} bars x {len(panel.symbols)} symbols "
      f"({panel.close.index[0]:%Y-%m-%d} → {panel.close.index[-1]:%Y-%m-%d})")

## 2 · Strategy overview — hyperparameters and search ranges

What each knob does, before any number crunching:

| param | role in the strategy | intuition |
|---|---|---|
| `band_mult` | entry stretch in ATRs below/above the midline | **the** knob: higher = rarer, deeper, better-paid dips |
| `band_window` | midline SMA (bars = minutes on 1m) | what "the mean" is; also the exit level |
| `trend_window` | regime SMA (bars) | which dips are safe to buy; must be > `band_window` |
| `atr_window` | ATR smoothing window (bars) | how fast the band adapts to a name's volatility |
| `horizon` | time stop (bars held) | how long to wait for a reversion that isn't coming |
| `direction` | long / short / both | fading rips in momentum names is the risky half; default long |
| `sl_mult` | stop-loss distance in ATRs | 0 = ride the dip out until midline/horizon; >0 caps the tail |
| `skip_open` / `skip_close` | entry blackout (minutes) | no fills into the open/close chaos; inert on 1d |
| `eod_exit` | flat at session close | intraday means intraday; inert on 1d |

The table below shows, per parameter: the **default** used by the single
backtest (§3), the hand-picked **sweep grid** (§5), and the **Optuna search
range** (§6). A `—` means the stage keeps the default.

In [ ]:
import pandas as pd

def _fmt_space(spec):
    """One readable line for an optuna.space entry."""
    if spec["type"] == "categorical":
        return " | ".join(map(str, spec["choices"]))
    tail = " (log)" if spec.get("log") else (f", step {spec['step']}" if spec.get("step") else "")
    return f"{spec['low']} – {spec['high']}{tail}"

pd.DataFrame({
    "default": pd.Series(strat.params),
    "sweep grid (§5)": pd.Series({k: ", ".join(map(str, v)) for k, v in strat.sweep.items()}),
    "optuna search range (§6)": pd.Series({k: _fmt_space(s)
                                           for k, s in strat.optuna["space"].items()}),
}).fillna("—")

## 3 · Single backtest with the config defaults

Engine rules (identical for every strategy): signals execute at the **next
bar's adjusted open**, realistic costs are always charged, every symbol trades
its own starting cash.

> **Capital model:** ONE shared \$10k wallet (engine cash-shared mode) — each entry deploys `cash_frac` of the *free* cash, so the book is never more than 100% invested. The repo-wide comparability rule; see `notebooks/04_compare.ipynb` for the cross-strategy scoreboard.

In [ ]:
import pandas as pd
pf = bt.run_strategy(strat, panel, settings=settings)
stats = bt.portfolio_stats(pf)
fees, slippage = bt.panel_costs(settings.costs, panel)
stats |= bt.active_stats(pf, panel, fees=fees, slippage=slippage,
                         fill=strat.fill, total_return=stats["total_return"])
pd.Series(stats).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity, config defaults").show()

## 4 · Inspect a symbol — trades and band overlays

`inspect_symbol` is the microscope of this notebook, used twice: here on the
most-traded symbol with the *default* parameters, and again in §10 on the
*optimized* ones. How to read the figure:

* **Price panel** — the SMA midline with its ±`band_mult`-ATR Keltner bands
  (drawn from a close-to-close range proxy; signals use the true high/low
  ATR) plus the slower trend SMA. ▲ blue = long entry on a close under the
  lower band in an uptrend, ▼ orange = short entry over the upper band in a
  downtrend, ✕ = exit (midline touch, `horizon` time stop, or the optional
  ATR stop); the entry→exit segment is green when the trade won, red when it
  lost.
* **Momentum panel** — close vs the trend SMA: positive (blue) is where longs
  were allowed, negative (red) where shorts were.
* **Volume panel** — context only; this strategy does not read volume.

In [ ]:
def inspect_symbol(sym, pf, params):
    """Deep-dive one symbol of a portfolio: annotated chart + its trade table."""
    all_trades = bt.trade_records(pf)
    merged = {**strat.params, **params, "bar": panel.bar, "tz": panel.tz}
    overlays = (strat.indicators(panel.close[sym], panel.volume[sym], **merged)
                if strat.indicators else None)
    plots.plot_symbol(panel.close[sym], panel.volume[sym], all_trades, symbol=sym,
                      overlays=overlays, equity=pf.value(group_by=False)[sym]).show()
    mine = all_trades[all_trades["symbol"] == sym]
    return mine.sort_values("entry_ts").reset_index(drop=True).round(3)

trades = bt.trade_records(pf)
busiest = trades["symbol"].value_counts().index[0]
inspect_symbol(busiest, pf, strat.params)

## 5 · Grid sweep — a coarse map of the parameter space

Before optimizing, sweep the small hand-picked grid from `config.yaml`. The
ranked bars show the best single combos; the **heatmap grid** below them is the
multi-objective view — the same parameter pair judged by Sharpe, total return
*and* breadth (`pct_profitable` = share of symbols that made money) at once, so
a region that pays on return but is carried by one name stands out immediately.

In [ ]:
grid = bt.grid_from_config(strat.sweep, strat.valid)
print(f"{len(grid)} combos")
sweep_stats = bt.sweep(strat, panel, grid, settings=settings)
sweep_stats.sort_values("excess_cagr", ascending=False).head(8).round(3)

In [ ]:
plots.plot_sweep(sweep_stats).show()
plots.plot_heatmap_grid(
    sweep_stats,
    pairs=[("band_mult", "band_window"), ("trend_window", "horizon")],
    metrics=("sharpe", "total_return", "pct_profitable"),
).show()

## 6 · Optimize (resumable, runs outside the notebook)

`jobs.run_optimize` hands the work to a `tsim optimize` subprocess. The study
scores **`excess_cagr` (maximized) against `volatility` (minimized)** — how much
more per year than an equal-weight buy-and-hold of the same panel, and how bumpy the
book's own ride is (its annual risk rate) — so excess is above zero only by *beating
the market*, not by being long in a rising one; both annualized, so windows of
different lengths stay comparable. The pick only ever chooses a front trial that
beats buy-and-hold. The result is a
*Pareto front* — the set of parameter sets where improving one objective must
hurt the other. Every finished trial is checkpointed to `studies/{study}.log`;
**re-running this cell resumes** exactly where it stopped (Ctrl-C / kernel
restarts lose nothing).

While it runs, every finished trial streams one line below — its objectives,
the current front size, and its parameters — so you can watch the search move.
For a richer live view, copy the **`uvx optuna-dashboard …`** command from the
first output line into a second terminal: it serves parameter importances,
objective history and the front at http://127.0.0.1:8080 while the study runs.

In [ ]:
N_TRIALS = 1500   # raise for a deeper search; re-runs accumulate into the same study
jobs.run_optimize("keltner_reversion", n_trials=N_TRIALS)

## 7 · Read the study: Pareto front and the picked trial

Gray dots are all completed trials; the blue **staircase** is the exact
boundary of the dominated region (each labeled point is one Pareto-optimal
trial); the orange star is the trial `pick_best` selects under the weights
below. Hover any point for its full parameter set.

**Journals are stamped with their objectives.** The objectives changed from absolute
`(sharpe, total_return)` to `(info_ratio, excess_cagr)`, then `(excess_cagr ↑,
tracking_error ↓)`, and now `(excess_cagr ↑, volatility ↓)`. `opt.load_study` refuses a journal scored on anything but the
current pair instead of silently mislabelling it — bump `optuna.study` in
`config.yaml` for a clean start.

> **Note:** new trials are floor-pruned in-objective via `optuna.min_trades` in `config.yaml`; the admissibility filter below still guards older trials in resumed journals. The study name was bumped when the capital model changed to the shared wallet — earlier journals scored independent per-symbol books and are not comparable.

In [ ]:
study = opt.load_study(strat)
front = opt.pareto_frame(study)
everything = opt.trials_frame(study)

# Objectives are annualized: excess_cagr (annualized edge over equal-weight buy-and-hold,
# maximized) against volatility (the book's own annual risk rate, minimized).
# Two guards before picking: (1) an admissibility floor on train
# trades — a handful of lucky fills can post a huge excess_cagr that means nothing;
# (2) `opt.pick_from_front`, the ONE picking rule in the repo (opt.PICK_WEIGHTS, balanced),
# which only picks rows that beat buy-and-hold and collapses tied outcomes first so a
# coin flip between identical-scoring trials cannot masquerade as a decision.
# Watch `n_tied`: > 1 means the pick was arbitrary among that many equal strategies.
MIN_TRADES = 300      # admissibility floor for TRAIN trades — below this, ranking is noise

eligible = front[front["n_trades"] >= MIN_TRADES]
if eligible.empty:
    print(f"!! no Pareto trial reaches {MIN_TRADES} train trades — showing the raw front; "
          "treat any pick as inadmissible")
    eligible = front
objs = list(opt.OBJECTIVES)          # downstream cells rank the runner-ups by objs[0]
best_row = opt.pick_from_front(eligible)
best = next(t for t in study.best_trials if t.number == int(best_row["trial"]))
print(f"front: {len(front)} trials, {len(eligible)} admissible (>= {MIN_TRADES} trades), "
      f"{int(best_row['front_distinct'])} distinct outcomes; picked trial {best.number}"
      + (f" — TIED with {int(best_row['n_tied']) - 1} other(s), so this pick is arbitrary"
         if best_row["n_tied"] > 1 else "")
      + ("" if best_row["beats_benchmark"] else " — NOTHING on the front beats buy-and-hold"))
plots.plot_pareto(front, everything, picked=best.number).show()
eligible.head(8).round(3)

## 8 · Boundary check — did the optimum hit the search-space edge?

A picked trial sitting **on the edge of a searched range** is a hint that the
true optimum may lie outside it. `optimize.boundary_report` compares the
picked parameters (and the whole Pareto front) with `search:` ranges from
`config.yaml`: any numeric parameter within 5% of a bound — measured on the
searched scale, so `log: true` ranges are judged in log space — is flagged
with the bound to widen. Categorical parameters have no edges and are skipped.

If something is flagged: widen that bound in `config.yaml`, re-run §6 (the
same study name resumes, and the sampler explores the new region; rename the
study to restart clean), then re-read §7–§8 until the front sits inside the
box. `tsim optimize` prints the same check at the end of every CLI run.

In [ ]:
report = opt.boundary_report(best.params, strat.optuna["space"], front=front)
flagged = report[report["action"] != "ok"]
print("all clear — the searched ranges bracket the optimum" if flagged.empty
      else f"{len(flagged)} parameter(s) pressing against the search space:")
report

## 9 · Re-run the full backtest with the picked parameters

In [ ]:
print("picked params:", best.params)
pf_best = bt.run_strategy(strat, panel, best.params, settings=settings)
stats_best = bt.portfolio_stats(pf_best)
stats_best |= bt.active_stats(pf_best, panel, fees=fees, slippage=slippage,
                              fill=strat.fill, total_return=stats_best["total_return"])
pd.Series(stats_best).round(4)

In [ ]:
plots.plot_equity(bt.equity_curve(pf_best, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — equity with picked parameters").show()

## 10 · Verification — per-symbol deep dive

Portfolio means can hide an edge carried by a single name, so verify the
picked parameters symbol by symbol. First the full per-symbol breakdown of the
optimized backtest; then set `SYMBOL` in the next cell to any name that looks
interesting (the big winner, the one loser, the one with suspiciously few
trades) and re-run it to get the same annotated chart and trade table as §4 —
entries, exits, direction and the channel each breakout escaped.

In [ ]:
bt.symbol_stats(pf_best).round(4)

In [ ]:
SYMBOL = None   # ← set to any symbol from the table above (e.g. "NVDA"); None = most-traded
sym = SYMBOL or bt.trade_records(pf_best)["symbol"].value_counts().index[0]
inspect_symbol(sym, pf_best, best.params)

## 11 · Out-of-sample check — the tail the optimizer never saw

Everything above ran on data up to `data.end`, the **train cutoff**. This
section replays the picked parameters on the untouched tail — the one number
in this notebook the optimizer could not have fit.

The mechanics matter: signals are computed on the **full** history (so a
300-bar `trend_window` is warmed up on the day the holdout starts) and only
then are the panel and the signal frames sliced to the holdout rows, so every
metric below covers exclusively bars after the cutoff. Same symbols as the
train panel — re-resolving coverage over the holdout would quietly change the
universe. Expect worse numbers than §9; that is the point. Roughly comparable
Sharpe/breadth = the edge generalizes; a collapse to zero (or below) = §9 was
curve-fitting.

In [ ]:
from dataclasses import replace

assert d.get("end"), "set data.end in config.yaml to keep a holdout"
MIN_OOS_TRADES = 50
cutoff = pd.Timestamp(d["end"], tz="UTC")
full = load_panel(engine, panel.symbols, d["bar"], d["start"], None,
                  session=d.get("session", "regular"), exchanges=settings.exchanges)
after = full.close.index > cutoff
assert after.any(), "no bars stored after data.end — backfill the holdout tail first"
fees, slippage = bt.panel_costs(settings.costs, panel)
oos_panel = replace(full, close=full.close[after], open_=full.open_[after],
                    high=full.high[after], low=full.low[after], volume=full.volume[after])

def oos_portfolio(params):
    """Replay one parameter set on the untouched tail (warm-up sees history)."""
    sig = strat.signals(full, **{**strat.params, **params})
    oos_sig = {k: (v[after] if isinstance(v, pd.DataFrame) else v) for k, v in sig.items()}
    return bt.run(oos_panel, oos_sig, fees=fees, slippage=slippage,
                  init_cash=settings.backtest.init_cash, fill=strat.fill)

def oos_stats(params):
    """Holdout stats incl. the benchmark-relative pair, on the holdout slice."""
    pf = oos_portfolio(params)
    st = bt.portfolio_stats(pf)
    return pf, st | bt.active_stats(pf, oos_panel, fees=fees, slippage=slippage,
                                    fill=strat.fill, total_return=st["total_return"])

pf_oos, oos = oos_stats(best.params)
print(f"holdout: {full.close.index[after][0]:%Y-%m-%d} → {full.close.index[-1]:%Y-%m-%d} "
      f"({int(after.sum())} bars, {oos['n_trades']} trades)")
if oos["n_trades"] < MIN_OOS_TRADES:
    print(f"!! only {oos['n_trades']} OOS trades (< {MIN_OOS_TRADES}) — "
          "insufficient sample; do NOT read the metrics below as a verdict")
verdict = "BEATS" if oos["excess_cagr"] > 0 and oos["info_ratio"] > 0 else "does NOT beat"
print(f"hurdle: the pick {verdict} equal-weight buy-and-hold out of sample "
      f"(excess_cagr {oos['excess_cagr']:+.2%}, info_ratio {oos['info_ratio']:+.2f})")
plots.plot_equity(bt.equity_curve(pf_oos, init_cash=settings.backtest.init_cash),
                  title=f"{strat.name} — OUT-OF-SAMPLE equity, picked parameters").show()

# Directional consistency: replay the two next-best admissible front rows.
# Params come from the FrozenTrials, NOT the frame rows — iterrows() would
# upcast int params (trend_window/horizon/...) to floats and rolling() raises.
by_number = {t.number: t for t in study.best_trials}
table = {"picked": oos}
for trial_no in eligible.sort_values(objs[0], ascending=False)["trial"]:
    if int(trial_no) == best.number or len(table) >= 3:
        continue
    table[f"trial {int(trial_no)}"] = oos_stats(by_number[int(trial_no)].params)[1]
pd.DataFrame({"train (§9)": stats_best,
              **{f"holdout · {k}": v for k, v in table.items()}}).round(4)

## Read the result honestly

* §3–§10 are **in-sample**: the optimizer saw all the data they are judged on.
  §11 is the honest number — and even it gets stale after a few re-reads
  (each "check the holdout, tweak, check again" loop quietly fits the tail
  too). For a fresh view, re-run the pick on a *different universe*.
* Prefer combos where **`pct_profitable` is high** — an edge carried by one
  symbol is usually noise.
* More trials on the same data = more chances to overfit. The Pareto front
  going up does not mean the *strategy* got better.

**Add your own strategy:** copy `strategies/_template/`, write `signals()`, fill
in `config.yaml` — this exact notebook flow works unchanged. See the README's
"How to add a strategy".